In [43]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# load model
model = load_model("model_gru.h5")

model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (GRU)                (None, 20)                1680      
                                                                 
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 relu_0 (Activation)         (None, 64)                0         
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
Total params: 3089 (12.07 KB)
Trainable params: 3089 (12.07 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [44]:
import numpy as np

# input data
data = np.load('x_test.npy')

# first entry
first_entry = data[0]

print(first_entry)

[[0.15952477 0.42420992 0.50315732 0.15588839 0.01821551 0.5250904 ]
 [0.13198355 0.42420992 0.43044853 0.13066565 0.02555556 0.52938515]
 [0.12084481 0.42497075 0.49941784 0.11810011 0.01584965 0.52938515]
 [0.09772567 0.42650011 0.43417525 0.09645277 0.02305464 0.52938515]
 [0.07522886 0.42316338 0.43487811 0.07445391 0.02322459 0.45230561]
 [0.07168758 0.42502546 0.42783755 0.07097175 0.0269614  0.52938515]
 [0.04771096 0.42467484 0.50467014 0.04661617 0.01898072 0.45230561]
 [0.03010481 0.42227659 0.41231856 0.03006944 0.03670086 0.        ]
 [0.02749825 0.42577225 0.50376147 0.02686404 0.01826115 0.5250904 ]
 [0.02615888 0.45495465 0.57882947 0.02658588 0.07357438 0.45230561]
 [0.02448627 0.42638612 0.42848343 0.02420743 0.02644895 0.54769439]
 [0.02132582 0.41896281 0.44489309 0.02112756 0.01990387 0.54769439]
 [0.0185497  0.42124251 0.41241431 0.01853838 0.03692752 0.54769439]
 [0.01843033 0.42854261 0.50935459 0.0180016  0.02160535 0.45230561]
 [0.01834247 0.43120152 0.53034556

In [45]:
import numpy as np

# bin = np.binary_repr(num, width=width)

# converts decimal number as shows up in modelsim to actual decimal number
# since verilog implementation uses fixed point and python does not
def fixed_to_dec(num, nfrac):
    return num / (2 ** nfrac)

# vice versa
def dec_to_fixed(num, nfrac):
    return num * (2 ** nfrac)

In [46]:
import tensorflow as tf

# grab GRU layer from overall model
gru = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("layer1").output
)

# constant -0.0009765625 input
x = np.full((1, 20, 6), -0.0009765625, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for -1 input:", gru_output)

# constant 0 input
x = np.full((1, 20, 6), 0, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for 0 input:", gru_output)

# real input

x = np.expand_dims(first_entry, axis=0)

gru_output = gru.predict(x)
print("GRU output for actual input:", gru_output)

1/1 [==============================] - 0s 307ms/step
GRU output shape: (1, 20)
GRU output for -1 input: [[ 5.0905389e-03  1.6728804e-01  6.4357883e-01  2.5470734e-02
  -4.3091021e-02 -2.0401792e-05 -1.6998371e-02 -7.2253215e-01
   8.5067934e-01  2.3964050e-01 -1.8604267e-01  6.3019526e-01
   7.2923297e-01  9.4031227e-01 -3.4397484e-03 -3.2461387e-01
   1.3428175e-01  2.7717035e-02  4.5356870e-01  2.3317397e-01]]
1/1 [==============================] - 0s 18ms/step
GRU output shape: (1, 20)
GRU output for 0 input: [[ 0.00638667  0.16483407  0.6490506   0.01878716 -0.0463143  -0.00479441
  -0.0136489  -0.7286906   0.85105824  0.2505707  -0.17842358  0.63234377
   0.7343067   0.93981105 -0.00605593 -0.32525256  0.13518424  0.02427531
   0.45254812  0.23819612]]
1/1 [==============================] - 0s 18ms/step
GRU output for actual input: [[ 0.21623185 -0.10674865 -0.13893494 -0.20779276  0.39148578 -0.12147414
  -0.05005098 -0.13862137 -0.12582986  0.17577007  0.13803534  0.18737906
  -

In [47]:
# manually run through GRU computation

import numpy as np

# =======================
# set config
# =======================
TIMESTEPS = 20
INPUT_SIZE = 6
HIDDEN_SIZE = 20

# =======================
# helper functions
# =======================
def load_vector(filename):
    return np.loadtxt(filename)

def load_matrix(filename, rows, cols):
    data = np.loadtxt(filename)
    return data.reshape(rows, cols)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =======================
# get GRU weights
# =======================

gru = model.layers[0]

kernel, recurrent_kernel, bias = gru.get_weights()

# split kernels
W_z, W_r, W_h = np.split(kernel, 3, axis=1)
U_z, U_r, U_h = np.split(recurrent_kernel, 3, axis=1)

# split bias into input and recurrent
b_input = bias[0]   # (360,)
b_rec   = bias[1]   # (360,)

# split per gate (update, reset, candidate hidden state)
b_z, b_r, b_h = np.split(b_input, 3)
b_z_rec, b_r_rec, b_h_rec = np.split(b_rec, 3)

# =======================
# set input
# =======================
x = np.expand_dims(first_entry, axis=0)

# =======================
# initialize hidden state
# =======================
h = np.zeros(HIDDEN_SIZE) # more zeroes

# trace storage
z_trace = []
z_raw_trace = []
r_trace = []
r_raw_trace = []
h_tilde_trace = []
h_tilde_raw_trace = []
h_tilde_raw_U_trace = []
h_tilde_raw_W_trace = []
h_trace = []

# =======================
# GRU calculation
# =======================
for t in range(TIMESTEPS):
    x_t = x[0][t]

    # gates
    z_raw = x_t @ W_z + h @ U_z + b_z + b_z_rec
    r_raw = x_t @ W_r + h @ U_r + b_r + b_r_rec
    z = sigmoid(z_raw)
    r = sigmoid(r_raw)

    # candidate (reset_after=True)
    h_tilde_raw = x_t @ W_h + b_h + r * (h @ U_h + b_h_rec)
    # h_tilde_raw_U = h_t-1 * U_h + b_h_rec
    h_tilde_raw_U = h @ U_h + b_h_rec
    # h_tilde_raw_W = W_h * x_t + b_h
    h_tilde_raw_W = x_t @ W_h + b_h
    h_tilde = np.tanh(h_tilde_raw)

    # hidden state update
    h = (1 - z) * h_tilde + z * h

    # save traces
    z_trace.append(z)
    z_raw_trace.append(z_raw)
    r_trace.append(r)
    r_raw_trace.append(r_raw)
    h_tilde_trace.append(h_tilde)
    h_tilde_raw_trace.append(h_tilde_raw)
    h_tilde_raw_U_trace.append(h_tilde_raw_U)
    h_tilde_raw_W_trace.append(h_tilde_raw_W)
    h_trace.append(h)

# convert to arrays
z_raw_trace = np.array(z_raw_trace)
z_trace = np.array(z_trace)
r_raw_trace = np.array(r_raw_trace)
r_trace = np.array(r_trace)
h_tilde_trace = np.array(h_tilde_trace)
h_tilde_raw_trace = np.array(h_tilde_raw_trace)
h_tilde_raw_U_trace = np.array(h_tilde_raw_U_trace)
h_tilde_raw_W_trace = np.array(h_tilde_raw_W_trace)
h_trace = np.array(h_trace)

# =======================
# outputs
# =======================
print("final hidden state (first 10 units):")
print(h[:10])

print("\nhidden state evolution (first unit):")
print(h_trace[:, 0])

final hidden state (first 10 units):
[ 0.21623189 -0.10674868 -0.13893487 -0.2077928   0.39148581 -0.12147414
 -0.05005095 -0.13862148 -0.1258299   0.1757702 ]

hidden state evolution (first unit):
[-0.00578635 -0.00537235 -0.00709887 -0.00347156  0.00154734  0.01332501
  0.02352847  0.0314318   0.05156715  0.07415733  0.11091636  0.12830093
  0.1411169   0.14376876  0.15403042  0.16716109  0.18310157  0.18712887
  0.20031125  0.21623189]


In [48]:
# print detailed evolution of what signals should look like in modelsim

print("evolution in modelsim (by timestep):\n")

num_steps = min(100, r_trace.shape[1])
nfrac = 10

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = dec_to_fixed(r_trace[t, :5], nfrac)
    z_fix = dec_to_fixed(z_trace[t, :5], nfrac)
    h_tilde_fix = dec_to_fixed(h_tilde_trace[t, :5], nfrac)
    h_fix = dec_to_fixed(h_trace[t, :5], nfrac)
    
    r_raw = dec_to_fixed(r_raw_trace[t, :5], nfrac)
    z_raw = dec_to_fixed(z_raw_trace[t, :5], nfrac)
    h_tilde_raw = dec_to_fixed(h_tilde_raw_trace[t, :5], nfrac)
    h_tilde_raw_U = dec_to_fixed(h_tilde_raw_U_trace[t, :5], nfrac)
    h_tilde_raw_W = dec_to_fixed(h_tilde_raw_W_trace[t, :5], nfrac)
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw_U : {h_tilde_raw_U}")
    print(f"  h_tilde_raw_W : {h_tilde_raw_W}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution in modelsim (by timestep):

t = 0
  r_raw   : [-122.70936471 -193.14363665  -72.7506713    52.10826112  217.2876663 ]
  r       : [481.35931682 463.85673625 493.81997846 525.02425489 566.11900207]
  z_raw   : [  579.62429303 -1175.50943155   448.52933315  -364.4025389
   447.9495666 ]
  z       : [653.15714475 246.64287419 622.37327648 421.848742   622.23506209]
  h_tilde_raw_U : [-28.81407356  27.45491028  71.3564682   68.62835693 -78.93804932]
  h_tilde_raw_W : [  -2.81772566   44.51121226 -353.88958154   -5.12156146  285.19107177]
  h_tilde_raw : [ -16.36257211   56.94787737 -319.47820499   30.06550101  241.55012479]
  h_tilde : [ -16.36117963   56.88924    -309.50070848   30.05686457  237.1674135 ]
  h       : [  -5.92522126   43.18677353 -121.3903862    17.67458869   93.05229605]

t = 1
  r_raw   : [ -96.20356902 -232.08181146   19.04844889 -111.79499195 -576.03654351]
  r       : [487.96678233 454.22663828 516.76197491 484.07897938 371.67201681]
  z_raw   : [  367.84713

In [49]:
# print detailed evolution of what signals should look like

print("evolution (by timestep):\n")

num_steps = min(15, r_trace.shape[1])
nfrac = 10

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = r_trace[t, :5]
    z_fix = z_trace[t, :5]
    h_tilde_fix = h_tilde_trace[t, :5]
    h_fix = h_trace[t, :5]
    
    r_raw = r_raw_trace[t, :5]
    z_raw = z_raw_trace[t, :5]
    h_tilde_raw = h_tilde_raw_trace[t, :5]
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution (by timestep):

t = 0
  r_raw   : [-0.11983336 -0.18861683 -0.07104558  0.05088697  0.21219499]
  r       : [0.47007746 0.45298509 0.48224607 0.512719   0.55285059]
  z_raw   : [ 0.56603935 -1.14795843  0.43801693 -0.35586185  0.43745075]
  z       : [0.63784877 0.24086218 0.6077864  0.41196166 0.60765143]
  h_tilde_raw : [-0.01597907  0.05561316 -0.31199043  0.02936084  0.23588879]
  h_tilde : [-0.01597771  0.0555559  -0.30224679  0.02935241  0.2316088 ]
  h       : [-0.00578635  0.04217458 -0.1185453   0.01726034  0.09087138]

t = 1
  r_raw   : [-0.0939488  -0.22664239  0.018602   -0.1091748  -0.56253569]
  r       : [0.47653006 0.4435807  0.50465037 0.47273338 0.36296095]
  z_raw   : [ 0.35922572 -2.20830754  0.68336024 -0.36877934  0.31074872]
  z       : [0.58885299 0.09900695 0.66448825 0.40883601 0.57706801]
  h_tilde_raw : [-0.00477946 -0.01036501 -0.24484679  0.05704941  0.22225011]
  h_tilde : [-0.00477942 -0.01036464 -0.24006849  0.0569876   0.21866164]
  h       :

In [50]:
x = np.full((1, 100, 3), 0, dtype=np.float32)

intermediate_output = x
for i, layer in enumerate(model.layers):
    intermediate_output = layer(intermediate_output)
    print(f"Layer {i} ({layer.name}) output: {[intermediate_output.numpy()[:5]]}")

ValueError: Input 0 of layer "layer1" is incompatible with the layer: expected shape=(None, None, 6), found shape=(1, 100, 3)